In [ ]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from mylibs import *
#from json_csv_final_test import *
import os
import requests
import pandas as pd
import time
#pip install selenium

In [16]:
#def get_link(category,max_page):
driver = webdriver.Chrome()
driver.maximize_window()

list_links=[]
for page in  range(1,15):
  url=f'https://hasaki.vn/danh-muc/cham-soc-da-mat-c4.html?p={page}'
  driver.get(url)
    #xac dinh vung chua san pham:
  product_container =driver.find_element(by=By.CLASS_NAME, value='ProductGrid__grid')

  a_tags=product_container.find_elements(by=By.CLASS_NAME, value='v3_thumb_common_sp')
  for a_tag in a_tags:
   link = a_tag.get_attribute('href')+'\n' #xuong dong khi luu file txt   
   #print(link) 
   list_links.append(link)
  #print(f'Hoan thanh trang {page}')
print(f'Tim thay {len(list_links)} link san pham')  
write_txt(f'links_final_test.txt', list_links)
driver.close()

Tim thay 560 link san pham


In [7]:
driver = webdriver.Chrome()
driver.maximize_window()

list_links=read_txt('links_final_test.txt')
#for i in range(0,len(list_links)
for i in range(len(list_links)):
    link=list_links[i].strip('\n')
    driver.get(link)
    time.sleep(1)
    
    #ma san pham 
    ID_tag=driver.find_element(by=By.XPATH, value = '//div[@class="pb-2.5 pr-2.5"]/div[@class="flex items-center gap-1 text-[#326E51] text-[13px]"]')
    product_id=ID_tag.text.split(' ')[-1]
    #print('ID: ',product_id)
    
    # ten san pham:
    name_tag=driver.find_element(by=By.TAG_NAME, value='h1')
    product_name=name_tag.text
    #print('Tên sản phẩm: ', product_name)

    
    #giá ban
    price_sale_tag=driver.find_element(by=By.CSS_SELECTOR, value='span.text-orange.text-lg.font-bold')
    product_price_sale=round(float(price_sale_tag.text.rstrip('₫').replace('.','')))
    #print('Giá bán: ',product_price_sale)
    
    #gia goc:
    try:
     origin_tag=driver.find_element(by=By.XPATH, value = '//div[@class="mb-1.5"]/div[@class="text-[13px]"]')
     product_price=round(float(origin_tag.text.split(' ')[3].replace('.','')))
    except:
     product_price = 'Null'
    #print('Giá gốc: ',product_price)

    # Phân loại:
    list_phan_loai=[]
    mota_tags=driver.find_elements(by=By.XPATH, value = '//div[@class="mb-[5px]"]/div/a')
    for mota_tag in mota_tags:
       product_mota1=mota_tag.text.rstrip()
       list_phan_loai.append(product_mota1)
       product_mota=mota_tag.get_attribute('title').rstrip()
       list_phan_loai.append(product_mota)
    #print(list_phan_loai)
       
        
    #Mô tả id DescriptionInfo
    try: 
     des_tag=driver.find_element(by=By.ID, value='DescriptionInfo')
     view_more_tag=des_tag.find_element(by=By.CSS_SELECTOR, value='div.absolute button')
     view_more_tag.click()
     time.sleep(0.3)
     product_des=des_tag.text  
    except:
     product_des = 'Null'  
    #print(product_des)
    
    #diem trung binh  
    Avg_tag=driver.find_element(by=By.XPATH, value = '//div[@class="w-[41.67%]"]/div/div')
    product_avg=Avg_tag.text
    #print(product_avg)
    
    #luu thong tin vao file json
    data_dir='data/final_test'
    if not os.path.exists(data_dir):
            os.makedirs(data_dir)
    file_name=f'{str(i+1).zfill(3)}.json' #001.json-560.json
    path=f'{data_dir}/{file_name}'
    dict_product={
     'Ma_san_pham' :product_id,
     'ten_san_pham':product_name, 
     'gia_ban' :product_price_sale, 
     'gia_goc' : product_price, 
     'Phan_loai': list_phan_loai,
     'Mo_ta': product_des,
     'Diem_danh_gia': product_avg
     }
    write_json(path, dict_product)
    print(f'da luu san pham: {file_name}')
driver.close()

da luu san pham: 124.json


In [4]:
 data_dir=f'data/final_test'
 data_export = 'data'
 list_product_ids=[]
 list_product_names=[]
 list_product_price_sales=[]
 list_product_prices=[]
 list_list_phan_loais=[]
 list_product_deses=[]
 list_product_avgs=[]
 for file_name in os.listdir(data_dir):
    if file_name.endswith('.json'):
        path=f'{data_dir}/{file_name}'
        #print(path)
        dict_product=read_json(path)
        list_product_ids.append(dict_product['Ma_san_pham'])
        list_product_names.append(dict_product['ten_san_pham'])
        list_product_price_sales.append(dict_product['gia_ban'])
        list_product_prices.append(dict_product['gia_goc'])
        str_phan_loais='\n'.join(dict_product['Phan_loai'])
        list_list_phan_loais.append(str_phan_loais)
        list_product_deses.append(dict_product['Mo_ta'])
        list_product_avgs.append(dict_product['Diem_danh_gia'])

 else:
    df_product=pd.DataFrame({
        'Ma_san_pham':list_product_ids,
        'ten_san_pham':list_product_names,
        'gia_ban':list_product_price_sales,
        'gia_goc':list_product_prices,
        'Phan_loai':list_list_phan_loais,
        'Mo_ta':list_product_deses,
        'Diem_danh_gia': list_product_avgs
     
    })
    df_product.to_csv(f'{data_export}/san_pham.csv', index=False)

In [5]:
df=pd.read_csv('data/san_pham.csv')
df.head()

,Ma_san_pham,ten_san_pham,gia_ban,gia_goc,Phan_loai,Mo_ta,Diem_danh_gia
0,318900012,Nước Hoa Hồng Klairs Không Mùi Cho Da Nhạy Cảm...,212000,435000,2x30ml\n\n2x180ml\n\n10ml\n\n30ml\n\n180ml\n\n...,Nước Hoa Hồng Klairs Supple Preparation là dòn...,4.8
1,422208973,Sữa Rửa Mặt CeraVe Sạch Sâu Cho Da Thường Đến ...,358000,475000,[Mini] Sữa Dưỡng Thể CeraVe Cho Da Khô Đến Rất...,Sữa Rửa Mặt Cerave Sạch Sâu là sản phẩm sữa rử...,4.9
2,205100137,"Nước Tẩy Trang L'Oreal Tươi Mát Cho Da Dầu, Hỗ...",133000,239000,2x400ml\n\n5x95ml\n\n95ml\n\n400ml\n\n\nLàm Sạ...,Nước Tẩy Trang L'Oréal là dòng sản phẩm tẩy tr...,4.8
3,253900006,Kem Chống Nắng Skin1004 Cho Da Nhạy Cảm SPF 50...,205000,445000,\n2x20ml\n\n20ml\n\n50ml,Kem Chống Nắng Skin1004 Cho Da Nhạy Cảm là sản...,4.8
4,422218358,Sữa Chống Nắng Anessa Dưỡng Da Kiềm Dầu 60ml (...,399000,715000,\n2x20ml\n\n2x60ml\n\n20ml\n\n60ml\n\n60ml+20m...,Sữa Chống Nắng Anessa Perfect UV Sunscreen Ski...,4.9


Ma_san_pham        int64
ten_san_pham      object
gia_ban            int64
gia_goc           object
Phan_loai         object
Mo_ta             object
Diem_danh_gia    float64
dtype: object